# Biển số Việt Nam — benchmark GPU Colab (detect + OCR + pipeline INT8)

Notebook này **không train**. Nó lấy `vn_alpr_results.zip` trên Drive, đúng cấu trúc unzip:

- ảnh detection: `canonical/rfdetr_data/{train,valid,test}` + `canonical/instances_*.json`
- crop OCR: `canonical/ocr_images` + `canonical/val.csv`
- ONNX: `rtdetrv4_s/`, `rfdetr_small/`, `dfine_*`, `cct_s_v2/`

Việc chạy: PTQ INT8, **AP detector**, **exact/CER OCR**, **pipeline detect→crop→OCR**, latency ảnh thật và latency ONNX giả.

[Open in Colab](https://colab.research.google.com/github/nqMinh-GR/NCS/blob/main/notebooks/VN_ALPR_Colab_INT8.ipynb)

Repo: `https://github.com/nqMinh-GR/NCS.git`


## 1. Lấy code


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/nqMinh-GR/NCS.git"
ROOT = Path("/content/NCS")
if (ROOT / ".git").exists():
    subprocess.check_call(["git", "-C", str(ROOT), "pull", "--ff-only"])
elif (Path.cwd() / "configs" / "project.yaml").exists():
    ROOT = Path.cwd()
else:
    subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(ROOT)])
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
for name in list(sys.modules):
    if name == "vn_alpr" or name.startswith("vn_alpr."):
        del sys.modules[name]
try:
    rev = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    rev = "unknown"
print("ROOT", ROOT.resolve(), "rev", rev)


## 2. Tham số

- `RESULTS_ZIP`: để `None` thì lần lượt tìm `MyDrive/vn_alpr/vn_alpr_results.zip`, `MyDrive/NCS/vn_alpr_results.zip`, `MyDrive/vn_alpr_results.zip`.
- `MAX_IMAGES`: `None` chấm hết val (detection + OCR + pipeline). Đặt số nguyên để chạy ngắn.
- `CALIB_SAMPLES`: ảnh train trong zip dùng cho PTQ INT8.
- Latency ONNX giả (tensor 0) tách với latency ảnh thật đo trong bước detect/OCR/pipeline.


In [ ]:
from pathlib import Path

RESULTS_ZIP = None  # Path hoặc None = tìm trên Drive
EXTRACT_DIR = Path("/content/vn_alpr_results")
OUT_DIR = Path("/content/vn_alpr_int8")
SPLIT = "val"
MAX_IMAGES = None  # ví dụ 100 nếu muốn chạy nhanh
CALIB_SAMPLES = 256

LATENCY_WARMUP = 20
LATENCY_ITERATIONS = 100
LATENCY_SECONDS = 10
LATENCY_REPEATS = 1


## 3. Cài ONNX Runtime GPU


In [ ]:
import subprocess, sys

def pip_install(packages):
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", *packages]
    print(" ".join(cmd))
    completed = subprocess.run(cmd, check=False)
    return completed.returncode == 0

pip_install(["PyYAML", "Pillow", "numpy", "onnx"])
if not pip_install(["onnxruntime-gpu"]):
    print("onnxruntime-gpu missing, falling back to onnxruntime")
    pip_install(["onnxruntime"])
import onnxruntime as ort
print("providers", ort.get_available_providers())


## 4. Mount Drive và giải nén zip


In [ ]:
from pathlib import Path
from google.colab import drive
from vn_alpr.eval_results import extract_results_zip, find_results_zip

drive.mount("/content/drive")
zip_path = Path(RESULTS_ZIP) if RESULTS_ZIP else find_results_zip()
print("zip", zip_path, "bytes", zip_path.stat().st_size)
RESULTS_DIR = extract_results_zip(zip_path, EXTRACT_DIR)
print("results", RESULTS_DIR)
canonical = RESULTS_DIR / "canonical"
print("val.csv", (canonical / "val.csv").is_file())
print("ocr_images", sum(1 for _ in (canonical / "ocr_images").glob("*.jpg")))
print("det val images", sum(1 for _ in (canonical / "rfdetr_data" / "valid").glob("*.jpg")))
print("instances_val", (canonical / "instances_val.json").is_file())


## 5. Pipeline đầy đủ: INT8 + detect + OCR + detect→OCR

Ô này gọi `run_int8_eval`. Trong đó:

1. PTQ INT8 từng ONNX, calibrate detector bằng ảnh `rfdetr_data/train`, OCR bằng crop train.
2. **Detection**: letterbox RF-DETR / resize D-FINE & RT-DETRv4, ra box xywh pixel, tính AP50, recall, precision trên `instances_val.json`.
3. **OCR**: crop `canonical/ocr_images` + `val.csv`, exact/CER.
4. **Pipeline**: detect → crop box → OCR (`cct_s_v2` nếu có). `chain_exact` chỉ trên ảnh val có nhãn chữ (khớp `sample_id` với `ocr.jsonl`).
5. Latency ảnh thật (p50 trên val) và latency tensor 0 (so sánh engine).


In [ ]:
import json
from vn_alpr.config import apply_train_overrides, load_project
from vn_alpr.eval_results import (
    discover_onnx,
    load_detection_split,
    load_ocr_rows,
    run_int8_eval,
)
from vn_alpr.hardware import write_probe

overrides = {
    "latency_warmup": LATENCY_WARMUP,
    "latency_iterations": LATENCY_ITERATIONS,
    "latency_seconds": LATENCY_SECONDS,
    "latency_repeats": LATENCY_REPEATS,
}
cfg = apply_train_overrides(load_project(), overrides)
cfg["quantization"]["calibration_samples"] = int(CALIB_SAMPLES)
print("onnx", [(m["model_id"], m["kind"]) for m in discover_onnx(RESULTS_DIR)])
print("ocr rows", len(load_ocr_rows(canonical, SPLIT, max_images=MAX_IMAGES)))
print("det rows", len(load_detection_split(canonical, SPLIT, max_images=MAX_IMAGES)))
hardware = write_probe(OUT_DIR)
report = run_int8_eval(
    RESULTS_DIR,
    OUT_DIR,
    cfg,
    hardware,
    split=SPLIT,
    max_images=MAX_IMAGES,
    prefer_cuda=True,
)
print("status", report["status"], "det", report["detection_images"], "ocr", report["ocr_images"])
print("--- detection AP50 ---")
for model_id, payload in report["detection"].items():
    fp32 = payload.get("fp32") or {}
    int8 = payload.get("int8") or {}
    print(model_id, "fp32_ap50", fp32.get("ap50"), "recall", fp32.get("recall_at_threshold"),
          "int8_ap50", int8.get("ap50"), "real_p50_ms", fp32.get("real_image_p50_ms"), "gate", payload.get("gate"))
print("--- OCR exact ---")
for model_id, payload in report["ocr"].items():
    fp32 = payload.get("fp32") or {}
    int8 = payload.get("int8") or {}
    print(model_id, "fp32_exact", fp32.get("exact_normalized"), "cer", fp32.get("cer"),
          "int8_exact", int8.get("exact_normalized"), "gate", payload.get("gate"))
print("--- pipeline detect+OCR ---")
for key, chain in report["pipeline"].items():
    print(key, "chain_exact", chain.get("chain_exact"),
          "ocr_given_det", chain.get("ocr_exact_given_detection"),
          "gt_box_ocr", (chain.get("ocr_on_gt_box") or {}).get("exact_normalized"),
          "e2e_p50_ms", chain.get("e2e_p50_ms"))


## 6. Bảng kết quả


In [ ]:
from pathlib import Path

for name in (
    "detector_int8_leaderboard.csv",
    "ocr_int8_leaderboard.csv",
    "pipeline_int8_leaderboard.csv",
    "fp32_latency.csv",
    "int8_latency.csv",
    "int8_quantization.csv",
    "recommendation.md",
):
    path = OUT_DIR / name
    print("=" * 8, name)
    print(path.read_text(encoding="utf-8")[:5000] if path.exists() else "missing")


## 7. Copy báo cáo sang Drive


In [ ]:
import shutil
from pathlib import Path

dest = Path("/content/drive/MyDrive/vn_alpr/int8_eval")
dest.mkdir(parents=True, exist_ok=True)
copied = []
for path in sorted(OUT_DIR.rglob("*")):
    if not path.is_file():
        continue
    if path.suffix.lower() in {".onnx"} and path.stat().st_size > 80 * 1024 * 1024:
        continue
    target = dest / path.relative_to(OUT_DIR)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, target)
    copied.append(str(target.relative_to(dest)))
print("copied", len(copied), "files to", dest)
print("leaderboard", dest / "ocr_int8_leaderboard.csv")
